# Machine Learning Preprocessing 

## Import libraries 

In [39]:
import pandas as pd 

## Load somatic cancer variants

In [40]:
variants = pd.read_csv(
  "/home/anekl/git/master/cancer_variants_annotation_pipeline/output/variants_with_maves.tsv",
  sep="\t", 
  low_memory=False
)

print(f"Loaded {len(variants)} variants.")

Loaded 1120406 variants.


## Extract columns for ML-modeling

In [53]:
print(variants.columns.tolist())

['Hugo_Symbol', 'Entrez_Gene_Id', 'Center', 'NCBI_Build', 'Chromosome', 'Start_Position', 'End_Position', 'Strand', 'Consequence', 'Variant_Classification', 'Variant_Type', 'Reference_Allele', 'Tumor_Seq_Allele1', 'Tumor_Seq_Allele2', 'dbSNP_RS', 'dbSNP_Val_Status', 'Tumor_Sample_Barcode', 'Matched_Norm_Sample_Barcode', 'Match_Norm_Seq_Allele1', 'Match_Norm_Seq_Allele2', 'Tumor_Validation_Allele1', 'Tumor_Validation_Allele2', 'Match_Norm_Validation_Allele1', 'Match_Norm_Validation_Allele2', 'Verification_Status', 'Validation_Status', 'Mutation_Status', 'Sequencing_Phase', 'Sequence_Source', 'Validation_Method', 'Score', 'BAM_File', 'Sequencer', 't_ref_count', 't_alt_count', 'n_ref_count', 'n_alt_count', 'HGVSc', 'HGVSp', 'HGVSp_Short', 'Transcript_ID', 'RefSeq', 'Protein_position', 'Codons', 'Exon_Number', 'gnomAD_AF', 'gnomAD_AFR_AF', 'gnomAD_AMR_AF', 'gnomAD_ASJ_AF', 'gnomAD_EAS_AF', 'gnomAD_FIN_AF', 'gnomAD_NFE_AF', 'gnomAD_OTH_AF', 'gnomAD_SAS_AF', 'FILTER', 'Polyphen_Prediction', 

In [54]:
wanted_columns = ['gnomAD_AF', 'Hotspot_Type', 'In_Hotspot', 'DOMAIN_NAME', 'IN_DOMAIN', 'FEATURE_TYPE', 'IN_FUNC_SITE', 'Germline_Proximity', 'MaveDB_score', 'ONCOGENIC']
ml_df = variants[wanted_columns].copy() 

In [55]:
print(f"The dataset was reduced from {variants.shape[1]} to {ml_df.shape[1]} columns.")
display(ml_df.head())

The dataset was reduced from 109 to 10 columns.


,gnomAD_AF,Hotspot_Type,In_Hotspot,DOMAIN_NAME,IN_DOMAIN,FEATURE_TYPE,IN_FUNC_SITE,Germline_Proximity,MaveDB_score,ONCOGENIC
0,0.000000,single residue,True,Ras,True,Binding site,True,0.0,-0.166043,Oncogenic
1,0.000004,single residue,True,PK_Tyr_Ser-Thr,True,NaN,False,1.0,NaN,Oncogenic
2,0.000028,single residue,True,PK_Tyr_Ser-Thr,True,Binding site;Topological domain,True,1.0,NaN,Oncogenic
3,0.000016,single residue,True,P53,True,Region,True,1.0,1.145501,Oncogenic
4,NaN,single residue,True,Ras,True,Binding site,True,0.0,NaN,Oncogenic


## Filter to wanted oncogenic classes 

In [60]:
wanted_classes = ['Oncogenic', 'Likely Neutral']
filtered = ml_df[ml_df['ONCOGENIC'].isin(wanted_classes)]

print(f"The dataset was reduced from {len(ml_df)} to {len(filtered)} variants.")

The dataset was reduced from 3107 to 3107 variants.


## Create new columns

In [61]:
ml_df['has_MAVE'] = ml_df['MaveDB_score'].notna()
print(ml_df.head())

ml_df['has_gnomAD_AF'] = ml_df['gnomAD_AF'].notna() 

   gnomAD_AF    Hotspot_Type  In_Hotspot     DOMAIN_NAME  IN_DOMAIN  \
0   0.000000  single residue        True             Ras       True   
1   0.000004  single residue        True  PK_Tyr_Ser-Thr       True   
2   0.000028  single residue        True  PK_Tyr_Ser-Thr       True   
3   0.000016  single residue        True             P53       True   
4   0.000000  single residue        True             Ras       True   

                      FEATURE_TYPE  IN_FUNC_SITE Germline_Proximity  \
0                     Binding site          True                0.0   
1                          Unknown         False                1.0   
2  Binding site;Topological domain          True                1.0   
3                           Region          True                1.0   
4                     Binding site          True                0.0   

   MaveDB_score  ONCOGENIC  has_MAVE  has_gnomAD_AF  
0     -0.166043  Oncogenic      True           True  
1           NaN  Oncogenic     False  

## Check NA-values 

In [62]:
print("Number of rows with missing values, per column:")
print(ml_df.isnull().sum())

Number of rows with missing values, per column:
gnomAD_AF                0
Hotspot_Type             0
In_Hotspot               0
DOMAIN_NAME              0
IN_DOMAIN                0
FEATURE_TYPE             0
IN_FUNC_SITE             0
Germline_Proximity       0
MaveDB_score          2824
ONCOGENIC                0
has_MAVE                 0
has_gnomAD_AF            0
dtype: int64


# Handle missing values 

In [63]:
# fill missing values in 'gnomAD_AF' column with zero 
ml_df['gnomAD_AF'] = ml_df['gnomAD_AF'].fillna(0) 

# fill missing values in 'Hotspot_Type', 'DOMAIN_NAME' and 'FEATURE_TYPE' with 'Unknown' 
type_cols = ['Hotspot_Type', 'DOMAIN_NAME', 'FEATURE_TYPE'] 
ml_df[type_cols] = ml_df[type_cols].fillna('Unknown')

# fill missing values in germline proxomity with '999' (high value) 
ml_df['Germline_Proximity'] = ml_df['Germline_Proximity'].fillna('999')

ml_df.head(10) 

,gnomAD_AF,Hotspot_Type,In_Hotspot,DOMAIN_NAME,IN_DOMAIN,FEATURE_TYPE,IN_FUNC_SITE,Germline_Proximity,MaveDB_score,ONCOGENIC,has_MAVE,has_gnomAD_AF
0,0.000000,single residue,True,Ras,True,Binding site,True,0.0,-0.166043,Oncogenic,True,True
1,0.000004,single residue,True,PK_Tyr_Ser-Thr,True,Unknown,False,1.0,NaN,Oncogenic,False,True
2,0.000028,single residue,True,PK_Tyr_Ser-Thr,True,Binding site;Topological domain,True,1.0,NaN,Oncogenic,False,True
3,0.000016,single residue,True,P53,True,Region,True,1.0,1.145501,Oncogenic,True,True
4,0.000000,single residue,True,Ras,True,Binding site,True,0.0,NaN,Oncogenic,False,True
5,0.000000,single residue,True,Unknown,False,Unknown,False,2.0,NaN,Oncogenic,False,True
8,0.000000,single residue,True,Unknown,False,Unknown,False,24.0,NaN,Oncogenic,False,True
16,0.000000,single residue,True,PH,True,Binding site,True,8.0,NaN,Oncogenic,False,True
18,0.000000,single residue,True,PK_Tyr_Ser-Thr,True,Topological domain,True,66.0,NaN,Oncogenic,False,True
20,0.000000,single residue,True,Ras,True,Unknown,False,10.0,NaN,Oncogenic,False,True
